# Lesson 11: Regression Discontinuity Design

## Opening Story: The Oregon Health Insurance Experiment

In 2008, Oregon expanded its Medicaid program through a lottery. Over 90,000 people applied for about 35,000 spots. The lottery created a natural experiment: those just above and below the cutoff had nearly identical characteristics, but different treatment probabilities.

This is the essence of regression discontinuity design (RDD): when treatment is assigned based on a cutoff, we can compare units just above and below the cutoff to estimate causal effects.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Explain the RDD framework
2. Implement sharp and fuzzy RDD
3. Conduct bandwidth selection
4. Test for manipulation of the running variable
5. Interpret local average treatment effects

---

## 11.1 The RDD Framework

### Sharp RDD

Treatment is deterministically assigned based on a cutoff:

$$T_i = \mathbb{1}(X_i \geq c)$$

The causal effect at the cutoff:

$$\tau = \lim_{x \downarrow c} E[Y_i | X_i = x] - \lim_{x \uparrow c} E[Y_i | X_i = x]$$

### Fuzzy RDD

Treatment is not perfectly determined by the cutoff, but there's a discontinuity in the probability of treatment:

$$P(T_i = 1 | X_i = x)$$

has a jump at $x = c$.

---

## 11.2 Sharp RDD Example

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.random.seed(42)
n = 1000

# Running variable (e.g., test score)
X = np.random.uniform(0, 100, n)

# Treatment assignment (sharp cutoff at 50)
cutoff = 50
T = (X >= cutoff).astype(int)

# Outcome (effect of 2 for those above cutoff)
Y = 10 + 0.5 * X + 2 * T + np.random.normal(0, 2, n)

# Fit separate regressions on each side
left_mask = X < cutoff
right_mask = X >= cutoff

model_left = LinearRegression()
model_left.fit(X[left_mask].reshape(-1, 1), Y[left_mask])

model_right = LinearRegression()
model_right.fit(X[right_mask].reshape(-1, 1), Y[right_mask])

# Estimate treatment effect at cutoff
effect_left = model_left.predict([[cutoff]])[0]
effect_right = model_right.predict([[cutoff]])[0]
effect_rdd = effect_right - effect_left

print(f"Effect at cutoff: {effect_rdd:.3f}")
print(f"True effect: 2.0")

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(X[left_mask], Y[left_mask], alpha=0.3, s=10, label='Control')
plt.scatter(X[right_mask], Y[right_mask], alpha=0.3, s=10, label='Treated')

x_range = np.linspace(0, 100, 100)
plt.plot(x_range[x_range < cutoff], model_left.predict(x_range[x_range < cutoff].reshape(-1, 1)),
         color='blue', linewidth=2, label='Control fit')
plt.plot(x_range[x_range >= cutoff], model_right.predict(x_range[x_range >= cutoff].reshape(-1, 1)),
         color='red', linewidth=2, label='Treated fit')

plt.axvline(x=cutoff, color='black', linestyle='--', label='Cutoff')
plt.xlabel('Running Variable')
plt.ylabel('Outcome')
plt.title('Sharp Regression Discontinuity Design')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---

## 11.3 Fuzzy RDD

In [ ]:
# Fuzzy RDD: treatment probability jumps at cutoff
prob_treat = 0.2 + 0.6 * (X >= cutoff).astype(int)
T_fuzzy = np.random.binomial(1, prob_treat)

# Reduced form
Y_fuzzy = 10 + 0.5 * X + 2 * T_fuzzy + np.random.normal(0, 2, n)

# Wald estimator for fuzzy RDD
# Effect = (jump in outcome) / (jump in treatment probability)
# This is similar to IV with the cutoff as the instrument

---

## 11.4 Bandwidth Selection

The choice of bandwidth involves a bias-variance tradeoff:
- **Narrow bandwidth**: Less bias, more variance
- **Wide bandwidth**: More bias, less variance

---

## 11.5 Common Mistakes

1. **Manipulation of running variable**: Test for bunching at the cutoff
2. **Wrong bandwidth**: Use optimal bandwidth selection methods
3. **Extrapolating beyond the cutoff**: RDD estimates effects only at the cutoff
4. **Ignoring covariates**: Use covariates to improve precision

---

## 11.6 Knowledge Check

### Multiple Choice

1. **RDD estimates the effect:**
   - A) At the cutoff only
   - B) For everyone
   - C) For units near the cutoff
   - D) Both A and C

2. **Sharp RDD assumes:**
   - A) Perfect compliance
   - B) No compliance
   - C) Partial compliance
   - D) Random compliance

3. **The running variable is:**
   - A) The treatment
   - B) The outcome
   - C) The variable determining treatment
   - D) A confounder

4. **Manipulation of the running variable:**
   - A) Is good
   - B) Violates the design
   - C) Has no effect
   - D) Is necessary

5. **Fuzzy RDD is similar to:**
   - A) OLS
   - B) IV
   - C) Matching
   - D) DiD

### Short Answer

6. **Explain why RDD gives causal estimates.**

7. **What is the difference between sharp and fuzzy RDD?**

8. **How can you test for manipulation of the running variable?**

9. **Why is the LATE interpretation important in fuzzy RDD?**

10. **Give an example of a policy that uses a cutoff for eligibility.**

---

## 11.7 Summary

1. **RDD** exploits cutoffs in treatment assignment
2. **Sharp RDD** has perfect compliance at the cutoff
3. **Fuzzy RDD** has imperfect compliance
4. **Bandwidth selection** is crucial for validity
5. **LATE** is the parameter of interest

---

## 11.8 Further Reading

- Lee, D.S. & Lemieux, T. (2010). "Regression Discontinuity Designs in Economics." *Journal of Economic Literature*.
- Cattaneo, M.D., Idrobo, N., & Titiunik, R. (2020). *A Practical Introduction to Regression Discontinuity Designs*. Cambridge University Press.